## Notebook 07 — Meta-Model (Stacking Ensemble)

### Objective

Combine the predictions of XGBoost, LightGBM, and LSTM
into a single stacking ensemble using Logistic Regression
as the meta-model.

---

### Why Stacking Works

Each base model sees the data differently:

  XGBoost  → snapshot of 17 features, gradient boosted trees

  LightGBM → same snapshot, leaf-wise growth, different splits

  LSTM     → 20-day sequence, captures temporal momentum

No single model is best at every pattern.
XGBoost may catch sharp RSI spikes that LSTM misses.
LSTM may catch slow-building volume trends that trees miss.

A meta-model learns WHEN to trust each model.
It does not average blindly — it learns the weights.

---

### Why Logistic Regression as the Meta-Model

The meta-model input is 6 probability columns.
These are already well-calibrated soft predictions.
A simple linear combiner is ideal here — not another tree.

Logistic Regression:
  - Is interpretable (we can read the coefficients)
  - Cannot overfit on 6 features with 18,200 rows
  - Trains in milliseconds
  - Gives us calibrated probabilities at the output

---

### Input to This Notebook

Three OOF arrays from notebooks 04, 05, 06:

  xgb_oof_probs.npy     shape (24180, 3)

  lgbm_oof_probs.npy    shape (24180, 3)

  lstm_oof_probs.npy    shape (24180, 3)

Three coverage masks:
  xgb_oof_coverage.npy  shape (24180,)

  lgbm_oof_coverage.npy shape (24180,)
  
  lstm_oof_coverage.npy shape (24180,)

One feature list:
  feature_cols.pkl      17 feature names

---

### Output of This Notebook

  meta_model.pkl        Trained Logistic Regression stacker

  meta_val_probs.npy    Ensemble probabilities on 2024 val set
  
  meta_val_preds.npy    Final predicted labels on 2024 val set

----------

### Cell 1 — Imports and Configuration

#### Why These Libraries

numpy        → load OOF arrays, build meta-feature matrix

pandas       → load processed data, align targets

joblib       → save and load sklearn models

sklearn      → LogisticRegression, metrics, compute_sample_weight

#### Constants Defined Here

MODELS_DIR   → where all model files live

PROCESSED_PATH → the feature CSV from notebook 03

TRAIN_END    → same boundary used in notebooks 04, 05, 06

VAL_START    → same boundary used in notebooks 04, 05, 06

Using the same boundaries here is mandatory.
If we shift even one day, the OOF arrays no longer
align with the dataframe rows and targets will mismatch.

In [1]:
# ============================================================
# CELL 1 - Imports and Configuration
# ============================================================
# Import all libraries needed for meta-model training
# Define all constants in one place
# ============================================================

import numpy  as np
import pandas as pd
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model        import LogisticRegression
from sklearn.metrics             import (balanced_accuracy_score,
                                         f1_score,
                                         classification_report,
                                         confusion_matrix)
from sklearn.utils.class_weight  import compute_sample_weight

# ── Paths ─────────────────────────────────────────────────
MODELS_DIR     = '../models/'
PROCESSED_PATH = '../data/processed/all_stocks_features.csv'

# ── Split boundary — must match notebooks 04 05 06 exactly ─
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-01'

# ── Class label map ────────────────────────────────────────
CLASS_NAMES = {0: 'Sell', 1: 'Hold', 2: 'Buy'}

os.makedirs(MODELS_DIR, exist_ok=True)

print('Environment ready')
print('=' * 50)
print(f'  Models dir     : {MODELS_DIR}')
print(f'  Processed path : {PROCESSED_PATH}')
print(f'  Train end      : {TRAIN_END}')
print(f'  Val start      : {VAL_START}')
print(f'  Classes        : {CLASS_NAMES}')

Environment ready
  Models dir     : ../models/
  Processed path : ../data/processed/all_stocks_features.csv
  Train end      : 2023-12-31
  Val start      : 2024-01-01
  Classes        : {0: 'Sell', 1: 'Hold', 2: 'Buy'}


--------------------

### Cell 2 — Load Processed Data and OOF Arrays

#### What We Load Here

1. all_stocks_features.csv

   The processed feature dataframe from notebook 03.

   We need this for the true target labels (y) and
   to align rows with the OOF arrays.

2. xgb_oof_probs.npy  / xgb_oof_coverage.npy

   lgbm_oof_probs.npy / lgbm_oof_coverage.npy

   lstm_oof_probs.npy / lstm_oof_coverage.npy

   OOF probability arrays from notebooks 04, 05, 06.

   Shape (24180, 3) — one row per training row.

   Coverage mask tells us which rows have valid predictions.

3. feature_cols.pkl

   The 17 feature names saved in notebook 04.
   
   Not used for meta-model training but kept for reference
   and for the Streamlit dashboard in notebook 08.

#### Why We Must Sort Before Loading

The OOF arrays were built on df_train sorted by
['Ticker', 'Date'] — the same order the dataframe
was in during notebooks 04, 05, 06.

If we load the CSV and sort differently here,
row i in the dataframe will not match row i in the
OOF array and targets will be completely misaligned.

Always sort by ['Ticker', 'Date'] immediately after loading.

In [2]:
# ============================================================
# CELL 2 - Load Processed Data and OOF Arrays
# ============================================================
# Load the feature dataframe and all OOF arrays
# Sort order must match notebooks 04 05 06 exactly
# ============================================================

# ── Load processed feature dataframe ──────────────────────
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Processed data loaded')
print('=' * 50)
print(f'  Shape      : {df.shape}')
print(f'  Date range : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'  Tickers    : {df["Ticker"].nunique()}')
print(f'  Target dist:\n{df["target"].value_counts().sort_index()}')

# ── Split into train and validation — same boundary as 04 05 06 ──
df_train = df[df['Date'] <= TRAIN_END].reset_index(drop=True)
df_val   = df[df['Date'] >= VAL_START].reset_index(drop=True)

print(f'\nTrain shape : {df_train.shape}')
print(f'Val   shape : {df_val.shape}')

# ── Load OOF probability arrays ───────────────────────────
xgb_oof_probs  = np.load(MODELS_DIR + 'xgb_oof_probs.npy')
lgbm_oof_probs = np.load(MODELS_DIR + 'lgbm_oof_probs.npy')
lstm_oof_probs = np.load(MODELS_DIR + 'lstm_oof_probs.npy')

# ── Load coverage masks ────────────────────────────────────
xgb_oof_coverage  = np.load(MODELS_DIR + 'xgb_oof_coverage.npy')
lgbm_oof_coverage = np.load(MODELS_DIR + 'lgbm_oof_coverage.npy')
lstm_oof_coverage = np.load(MODELS_DIR + 'lstm_oof_coverage.npy')

# ── Load feature columns ───────────────────────────────────
feature_cols = joblib.load(MODELS_DIR + 'feature_cols.pkl')

print('\nOOF arrays loaded')
print('=' * 50)
print(f'  XGB  probs shape  : {xgb_oof_probs.shape}')
print(f'  LGBM probs shape  : {lgbm_oof_probs.shape}')
print(f'  LSTM probs shape  : {lstm_oof_probs.shape}')
print(f'  XGB  coverage sum : {xgb_oof_coverage.sum()}')
print(f'  LGBM coverage sum : {lgbm_oof_coverage.sum()}')
print(f'  LSTM coverage sum : {lstm_oof_coverage.sum()}')
print(f'  Feature cols      : {len(feature_cols)}')

# ── Sanity: OOF array rows must match df_train rows ───────
assert len(xgb_oof_probs)  == len(df_train), 'XGB OOF length mismatch'
assert len(lgbm_oof_probs) == len(df_train), 'LGBM OOF length mismatch'
assert len(lstm_oof_probs) == len(df_train), 'LSTM OOF length mismatch'

print('\nRow alignment check passed')

Processed data loaded
  Shape      : (29100, 27)
  Date range : 2019-03-14 to 2024-12-20
  Tickers    : 20
  Target dist:
target
0.0     7043
1.0    12530
2.0     9527
Name: count, dtype: int64

Train shape : (24180, 27)
Val   shape : (4920, 27)

OOF arrays loaded
  XGB  probs shape  : (24180, 3)
  LGBM probs shape  : (24180, 3)
  LSTM probs shape  : (24180, 3)
  XGB  coverage sum : 20100
  LGBM coverage sum : 20100
  LSTM coverage sum : 18200
  Feature cols      : 17

Row alignment check passed


------------

### Cell 3 — Align Coverage Masks and Extract Targets

#### What We Do Here

Build a single valid_mask that is True only where
ALL THREE models have a prediction.

XGB  covered : 20,100 rows

LGBM covered : 20,100 rows

LSTM covered : 18,200 rows

valid_mask   = xgb_coverage AND lgbm_coverage AND lstm_coverage

             = 18,200 rows  (LSTM is the most restrictive)

#### Why We Intersect Instead of Union

If we used the union (any model has a prediction),
some rows would have predictions from XGB and LGBM
but zeros from LSTM.

Feeding zeros into the meta-model for LSTM columns
would corrupt the logistic regression — it would
treat zero as a meaningful probability instead of
a missing value.

Intersection guarantees every row has valid honest
probabilities from all 3 models.

#### Extracting True Labels

We extract y_train_meta from df_train using the
same valid_mask.

These are the true target labels the meta-model
will train against — the same targets the base
models were trained on, just filtered to the
18,200 rows where all 3 models overlap.

In [3]:
# ============================================================
# CELL 3 - Align Coverage Masks and Extract Targets
# ============================================================
# Build intersection mask where all 3 models have predictions
# Extract true labels aligned to that mask
# ============================================================

# ── Build intersection coverage mask ──────────────────────
valid_mask = (
    (xgb_oof_coverage  == 1) &
    (lgbm_oof_coverage == 1) &
    (lstm_oof_coverage == 1)
)

print('Coverage alignment')
print('=' * 50)
print(f'  XGB  covered rows : {xgb_oof_coverage.sum():,}')
print(f'  LGBM covered rows : {lgbm_oof_coverage.sum():,}')
print(f'  LSTM covered rows : {lstm_oof_coverage.sum():,}')
print(f'  Intersection      : {valid_mask.sum():,}')
print(f'  Rows excluded     : {(~valid_mask).sum():,}')

# ── Extract true labels for valid rows ─────────────────────
y_train_meta = df_train.loc[valid_mask, 'target'].astype(int).values

print(f'\nMeta-model training target')
print('=' * 50)
print(f'  Samples : {len(y_train_meta):,}')
print(f'  Classes : {np.unique(y_train_meta)}')
print(f'  Sell (0): {(y_train_meta == 0).sum():,}  '
      f'({(y_train_meta == 0).mean()*100:.1f}%)')
print(f'  Hold (1): {(y_train_meta == 1).sum():,}  '
      f'({(y_train_meta == 1).mean()*100:.1f}%)')
print(f'  Buy  (2): {(y_train_meta == 2).sum():,}  '
      f'({(y_train_meta == 2).mean()*100:.1f}%)')

# ── Sanity checks ──────────────────────────────────────────
assert valid_mask.sum() == 18200, \
    f'Expected 18200 valid rows, got {valid_mask.sum()}'
assert len(y_train_meta) == valid_mask.sum(), \
    'Target length does not match valid mask'
assert set(np.unique(y_train_meta)) == {0, 1, 2}, \
    'Missing class in targets — check feature engineering'

print('\nAll alignment checks passed')

Coverage alignment
  XGB  covered rows : 20,100
  LGBM covered rows : 20,100
  LSTM covered rows : 18,200
  Intersection      : 18,200
  Rows excluded     : 5,980

Meta-model training target
  Samples : 18,200
  Classes : [0 1 2]
  Sell (0): 4,883  (26.8%)
  Hold (1): 7,292  (40.1%)
  Buy  (2): 6,025  (33.1%)

All alignment checks passed


--------------------

### Cell 4 — Build Meta-Feature Matrix (6 columns)

#### What We Do Here

Stack the OOF probabilities from all 3 models
into a single matrix that the meta-model will train on.

Each model outputs 3 probabilities per row:

  [P(Sell), P(Hold), P(Buy)]

We take only 2 per model — dropping P(Buy).

#### Why We Drop P(Buy) — Rule 4

For any classifier, probabilities always sum to 1.0:

  P(Sell) + P(Hold) + P(Buy) = 1.0   always

This means P(Buy) is never independent:

  P(Buy) = 1 - P(Sell) - P(Hold)

If we keep all 3 columns per model, we introduce
perfect multicollinearity into the meta-model input.

The Logistic Regression design matrix becomes
rank-deficient and coefficients become unstable.

Dropping P(Buy) from each model removes the
redundancy while losing zero information.

#### Final Meta-Feature Matrix Shape

  2 columns × 3 models = 6 features
  18,200 rows

  Columns:
    0 — XGB_Sell

    1 — XGB_Hold

    2 — LGBM_Sell

    3 — LGBM_Hold

    4 — LSTM_Sell
    
    5 — LSTM_Hold

In [4]:
# ============================================================
# CELL 4 - Build Meta-Feature Matrix (6 columns, Rule 4)
# ============================================================
# Stack OOF probabilities from all 3 models
# Drop P(Buy) from each model to remove multicollinearity
# ============================================================

# ── Extract valid rows from each OOF array ─────────────────
xgb_valid  = xgb_oof_probs[valid_mask]   # shape (18200, 3)
lgbm_valid = lgbm_oof_probs[valid_mask]  # shape (18200, 3)
lstm_valid = lstm_oof_probs[valid_mask]  # shape (18200, 3)

# ── Build meta-feature matrix — drop col 2 (Buy) each model ─
# Rule 4: use only P(Sell) and P(Hold) per model
meta_train = np.column_stack([
    xgb_valid[:, 0],    # XGB  P(Sell)
    xgb_valid[:, 1],    # XGB  P(Hold)
    lgbm_valid[:, 0],   # LGBM P(Sell)
    lgbm_valid[:, 1],   # LGBM P(Hold)
    lstm_valid[:, 0],   # LSTM P(Sell)
    lstm_valid[:, 1],   # LSTM P(Hold)
])
# P(Buy) for each model = 1 - P(Sell) - P(Hold) — always implied

META_FEATURE_NAMES = [
    'XGB_Sell', 'XGB_Hold',
    'LGBM_Sell', 'LGBM_Hold',
    'LSTM_Sell', 'LSTM_Hold'
]

print('Meta-feature matrix built')
print('=' * 50)
print(f'  Shape   : {meta_train.shape}')
print(f'  Columns : {META_FEATURE_NAMES}')
print(f'\n  Sample rows (first 3):')
for i in range(3):
    print(f'    Row {i}: {meta_train[i].round(3)}')

# ── Verify probabilities are valid ────────────────────────
assert meta_train.shape == (18200, 6), \
    f'Expected (18200, 6), got {meta_train.shape}'
assert np.all(meta_train >= 0) and np.all(meta_train <= 1), \
    'Probabilities out of [0, 1] range'
assert not np.any(np.isnan(meta_train)), \
    'NaN values found in meta-feature matrix'

print('\nAll meta-feature checks passed')

Meta-feature matrix built
  Shape   : (18200, 6)
  Columns : ['XGB_Sell', 'XGB_Hold', 'LGBM_Sell', 'LGBM_Hold', 'LSTM_Sell', 'LSTM_Hold']

  Sample rows (first 3):
    Row 0: [0.408 0.289 0.218 0.184 0.424 0.263]
    Row 1: [0.081 0.473 0.076 0.469 0.424 0.264]
    Row 2: [0.266 0.299 0.183 0.3   0.423 0.264]

All meta-feature checks passed


---------------------

### Cell 5 — OOF Sanity Validation

#### What We Do Here

Before training the meta-model we validate that
the OOF probabilities from each base model are
meaningful and honest.

We check 3 things for each model:

1. Probabilities sum to 1.0

   Every row's 3 probabilities must sum to exactly 1.

   If they don't, the OOF arrays are corrupted.

2. Individual model OOF accuracy

   Convert each model's OOF probabilities to predicted
   labels (argmax) and compute F1 Macro and Balanced
   Accuracy against the true labels.

   This tells us how strong each base model is
   before stacking.

3. Agreement between models

   How often do all 3 models predict the same class?
   Low agreement is actually fine — it means each
   model is catching different patterns, which is
   exactly what makes stacking useful.

#### Why This Cell Exists

If any base model's OOF probabilities are garbage
(all 0.33, or all pointing to one class), the
meta-model will train on bad input and produce
bad output. Catching this here saves debugging later.

In [5]:
# ============================================================
# CELL 5 - OOF Sanity Validation
# ============================================================
# Validate OOF probabilities before training meta-model
# Check sum-to-1, individual model accuracy, model agreement
# ============================================================

# ── 1. Probabilities sum to 1.0 ───────────────────────────
print('Probability sum check')
print('=' * 50)
for name, arr in [('XGB', xgb_valid), ('LGBM', lgbm_valid), ('LSTM', lstm_valid)]:
    row_sums = arr.sum(axis=1)
    all_sum  = np.allclose(row_sums, 1.0, atol=1e-5)
    print(f'  {name}  sum-to-1 : {all_sum}  '
          f'(mean={row_sums.mean():.6f}, '
          f'max_dev={np.abs(row_sums - 1.0).max():.2e})')

# ── 2. Individual model OOF performance ───────────────────
print(f'\nIndividual model OOF performance')
print('=' * 50)
results = {}
for name, arr in [('XGB', xgb_valid), ('LGBM', lgbm_valid), ('LSTM', lstm_valid)]:
    preds   = arr.argmax(axis=1)
    f1_mac  = f1_score(y_train_meta, preds, average='macro')
    bal_acc = balanced_accuracy_score(y_train_meta, preds)
    results[name] = {'F1 Macro': f1_mac, 'Bal Acc': bal_acc}
    print(f'  {name}')
    print(f'    F1 Macro         : {f1_mac:.4f}')
    print(f'    Balanced Accuracy: {bal_acc:.4f}')
    print()

# ── 3. Model agreement ────────────────────────────────────
print('Model agreement')
print('=' * 50)
xgb_preds  = xgb_valid.argmax(axis=1)
lgbm_preds = lgbm_valid.argmax(axis=1)
lstm_preds = lstm_valid.argmax(axis=1)

all_agree  = ((xgb_preds == lgbm_preds) &
              (lgbm_preds == lstm_preds))
xgb_lgbm   = (xgb_preds == lgbm_preds)
xgb_lstm   = (xgb_preds == lstm_preds)
lgbm_lstm  = (lgbm_preds == lstm_preds)

print(f'  All 3 agree       : {all_agree.mean()*100:.1f}%')
print(f'  XGB  == LGBM      : {xgb_lgbm.mean()*100:.1f}%')
print(f'  XGB  == LSTM      : {xgb_lstm.mean()*100:.1f}%')
print(f'  LGBM == LSTM      : {lgbm_lstm.mean()*100:.1f}%')

# ── Sanity asserts ────────────────────────────────────────
for name, arr in [('XGB', xgb_valid), ('LGBM', lgbm_valid), ('LSTM', lstm_valid)]:
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-5), \
        f'{name} probabilities do not sum to 1.0'

print('\nAll OOF sanity checks passed')

Probability sum check
  XGB  sum-to-1 : True  (mean=1.000000, max_dev=8.94e-08)
  LGBM  sum-to-1 : True  (mean=1.000000, max_dev=2.22e-16)
  LSTM  sum-to-1 : True  (mean=1.000000, max_dev=1.19e-07)

Individual model OOF performance
  XGB
    F1 Macro         : 0.3735
    Balanced Accuracy: 0.3759

  LGBM
    F1 Macro         : 0.3737
    Balanced Accuracy: 0.3751

  LSTM
    F1 Macro         : 0.3578
    Balanced Accuracy: 0.3646

Model agreement
  All 3 agree       : 36.4%
  XGB  == LGBM      : 79.6%
  XGB  == LSTM      : 43.7%
  LGBM == LSTM      : 44.1%

All OOF sanity checks passed


------------------------

### Cell 6 — Train Meta-Model (Logistic Regression)

#### What We Do Here

Train a Logistic Regression on the 6-column
meta-feature matrix built in Cell 4.

The meta-model learns:

  - When XGB says Sell strongly, how much to trust it

  - When LSTM disagrees with both tree models, who wins

  - Which model is most reliable for each class

#### Why Logistic Regression

The input is already 6 well-calibrated probabilities.

A linear combiner is the right tool here — not another
tree, not a neural network.

Reasons:

  1. Only 6 features — a complex model would overfit

  2. 18,200 samples — more than enough for logistic regression

  3. Interpretable — we can read the coefficients directly

  4. Fast — trains in milliseconds

  5. Gives calibrated probability output at the end

#### Why L2 Regularisation (C=1.0)

L2 is a safeguard against any residual multicollinearity
between the 6 features. Even after dropping P(Buy),
XGB_Sell and LGBM_Sell will be correlated because
both models saw the same data.

C=1.0 is the sklearn default — moderate regularisation.
Not too strong (would ignore signal) not too weak
(would allow unstable coefficients).

#### Why class_weight='balanced'

Hold is 40% of rows, Sell is 27%, Buy is 33%.
Without balancing the meta-model would bias toward
predicting Hold. Balanced weighting corrects this
— same approach used in notebooks 04, 05, 06.

In [6]:
# ============================================================
# CELL 6 - Train Meta-Model (Logistic Regression)
# ============================================================
# Train Logistic Regression on 6-column meta-feature matrix
# Use L2 regularisation and balanced class weights
# ============================================================

# ── Train meta-model ───────────────────────────────────────
meta_model = LogisticRegression(
    C            = 1.0,        # L2 regularisation strength
    penalty      = 'l2',       # safeguard against multicollinearity
    class_weight = 'balanced', # correct for Hold majority class
    max_iter     = 1000,       # enough iterations to converge
    random_state = 42,
    solver       = 'lbfgs'
)

meta_model.fit(meta_train, y_train_meta)

# ── Training set performance ───────────────────────────────
train_preds = meta_model.predict(meta_train)
train_f1    = f1_score(y_train_meta, train_preds, average='macro')
train_bal   = balanced_accuracy_score(y_train_meta, train_preds)

print('Meta-model trained')
print('=' * 50)
print(f'  Model          : LogisticRegression')
print(f'  C              : 1.0  (L2)')
print(f'  Class weight   : balanced')
print(f'  Training rows  : {len(y_train_meta):,}')
print(f'  Features       : {meta_train.shape[1]}')
print()
print(f'  Train F1 Macro          : {train_f1:.4f}')
print(f'  Train Balanced Accuracy : {train_bal:.4f}')
print()
print('  Note: training scores are always optimistic')
print('  True performance is in Cell 8 — validation set')

# ── Convergence check ─────────────────────────────────────
print(f'\n  Converged in   : {meta_model.n_iter_} iterations')
assert meta_model.n_iter_[0] < 1000, \
    'Meta-model did not converge — increase max_iter'

print('\nMeta-model training complete')

Meta-model trained
  Model          : LogisticRegression
  C              : 1.0  (L2)
  Class weight   : balanced
  Training rows  : 18,200
  Features       : 6

  Train F1 Macro          : 0.3935
  Train Balanced Accuracy : 0.3955

  Note: training scores are always optimistic
  True performance is in Cell 8 — validation set

  Converged in   : [40] iterations

Meta-model training complete


-------------

### Cell 7 — Inspect Meta-Model Coefficients

#### What We Do Here

Read the learned coefficients of the Logistic Regression
and interpret what the meta-model has learned.

Each coefficient tells us:

  - Positive value → this feature pushes toward this class

  - Negative value → this feature pushes away from this class

  - Large magnitude → model relies heavily on this feature

  - Near zero → model largely ignores this feature

#### What Healthy Coefficients Look Like

Coefficients should be in a reasonable range.

As per Rule 4 from the project instructions:

  If any coefficient exceeds ±10, something is wrong.
  Extreme values like ±100 or ±800 indicate that
  multicollinearity was not properly handled.

With L2 regularisation and only 6 features,
we expect all coefficients to be well-behaved
and interpretable.

#### Why This Cell Matters

This is the interpretability check.
A black-box meta-model that we cannot inspect
gives us no confidence in production.

Reading the coefficients tells us:

  - Which base model the meta-model trusts 
  
  - Which class each model is best at predicting
  
  - Whether the stacking is doing something sensible
    or just averaging blindly

In [7]:
# ============================================================
# CELL 7 - Inspect Meta-Model Coefficients
# ============================================================
# Read and interpret the logistic regression coefficients
# Verify no extreme values — Rule 4 check
# ============================================================

# ── Build coefficient dataframe ────────────────────────────
coef_df = pd.DataFrame(
    meta_model.coef_,
    columns = META_FEATURE_NAMES,
    index   = ['Sell', 'Hold', 'Buy']
)

print('Meta-model coefficients')
print('=' * 50)
print(coef_df.round(4).to_string())

# ── Summary statistics ─────────────────────────────────────
print(f'\nCoefficient summary')
print('=' * 50)
print(f'  Max value  : {coef_df.values.max():.4f}')
print(f'  Min value  : {coef_df.values.min():.4f}')
print(f'  Max abs    : {coef_df.abs().values.max():.4f}')

# ── Which model is trusted most per class ─────────────────
print(f'\nStrongest signal per class')
print('=' * 50)
for cls in ['Sell', 'Hold', 'Buy']:
    row       = coef_df.loc[cls]
    top_feat  = row.abs().idxmax()
    top_val   = row[top_feat]
    print(f'  {cls:4s} → strongest feature : '
          f'{top_feat:12s}  coef={top_val:.4f}')

# ── Rule 4 assertion ──────────────────────────────────────
assert coef_df.abs().max().max() < 50, \
    'Coefficients are extreme — check for multicollinearity'

print('\nCoefficient sanity check passed')
print('No extreme values detected')

Meta-model coefficients
      XGB_Sell  XGB_Hold  LGBM_Sell  LGBM_Hold  LSTM_Sell  LSTM_Hold
Sell    0.1063   -0.0032    -0.3474    -0.5615    -0.6514    -1.1154
Hold    0.6501    0.9058    -0.2493     0.2894     0.9791     1.8295
Buy    -0.7565   -0.9026     0.5966     0.2721    -0.3277    -0.7141

Coefficient summary
  Max value  : 1.8295
  Min value  : -1.1154
  Max abs    : 1.8295

Strongest signal per class
  Sell → strongest feature : LSTM_Hold     coef=-1.1154
  Hold → strongest feature : LSTM_Hold     coef=1.8295
  Buy  → strongest feature : XGB_Hold      coef=-0.9026

Coefficient sanity check passed
No extreme values detected


---------------------

### Cell 8 — Evaluate on Validation Set (2024)

#### What We Do Here

Run the full stacking pipeline on the 2024 validation
set — data the entire pipeline has never seen.

Steps:
  1. Load the 3 saved base models
  2. Generate predictions on df_val for each model
  3. Build LSTM sequences per ticker using train tail
     as context window — same logic as notebook 06
  4. Stack into 6-column meta-feature matrix
  5. Pass through meta-model to get final predictions
  6. Evaluate with F1 Macro and Balanced Accuracy

#### Why We Rebuild LSTM Sequences Here

The LSTM needs 20 days of history before it can
predict day 1 of 2024. Those 20 days come from
the tail of the training set (Dec 2023).

We combine the last 20 rows of df_train per ticker
with df_val per ticker, scale using the saved scaler,
then build sequences — exactly as notebook 06 did.

#### Why We Align XGB and LGBM to LSTM Row Count

XGB and LGBM predict on all 4,920 val rows.
LSTM produces sequences for all val rows too
since we prepend 20 training rows as context.

We align all three to the same rows before
stacking into the meta-feature matrix.

#### The Validation Set Is Opened Only Once

This is the first and only time 2024 data is
evaluated. Results here are final — we do not
retune after seeing them.

In [8]:
# ============================================================
# CELL 8 - Evaluate on Validation Set (2024)
# ============================================================
# Generate base model predictions on 2024 val set
# Stack into meta-features and evaluate final ensemble
# ============================================================

from tensorflow import keras

# ── Load base models and scaler ────────────────────────────
xgb_model   = joblib.load(MODELS_DIR + 'xgb_model.pkl')
lgbm_model  = joblib.load(MODELS_DIR + 'lgbm_model.pkl')
lstm_model  = keras.models.load_model(MODELS_DIR + 'lstm_model.keras')
lstm_scaler = joblib.load(MODELS_DIR + 'lstm_scaler.pkl')
lstm_cfg    = joblib.load(MODELS_DIR + 'lstm_sequence_config.pkl')

LOOKBACK = lstm_cfg['lookback']

print('Base models loaded')
print('=' * 50)
print(f'  XGBoost  : {MODELS_DIR}xgb_model.pkl')
print(f'  LightGBM : {MODELS_DIR}lgbm_model.pkl')
print(f'  LSTM     : {MODELS_DIR}lstm_model.keras')
print(f'  Lookback : {LOOKBACK} days')

# ── XGBoost and LightGBM predictions on val set ───────────
X_val  = df_val[feature_cols].values
y_val  = df_val['target'].astype(int).values

xgb_val_probs  = xgb_model.predict_proba(X_val)
lgbm_val_probs = lgbm_model.predict_proba(X_val)

print(f'\nTree model val predictions')
print('=' * 50)
print(f'  XGB  val probs shape  : {xgb_val_probs.shape}')
print(f'  LGBM val probs shape  : {lgbm_val_probs.shape}')

# ── LSTM predictions on val set ───────────────────────────
# Build sequences per ticker using train tail as context
# Scale using the saved training scaler

def build_sequences_for_val(df_tr, df_vl, feature_cols,
                             scaler, lookback):
    X_list, y_list = [], []

    for ticker, grp_val in df_vl.groupby('Ticker'):
        grp_train = df_tr[df_tr['Ticker'] == ticker]\
                        .sort_values('Date')
        grp_val   = grp_val.sort_values('Date')

        # Prepend last 20 training rows as context window
        combined  = pd.concat([grp_train.tail(lookback),
                                grp_val])
        combined  = combined.reset_index(drop=True)

        features  = scaler.transform(
                        combined[feature_cols].values)
        targets   = combined['target'].values

        for i in range(lookback, len(combined)):
            X_list.append(features[i-lookback:i])
            y_list.append(targets[i])

    return np.array(X_list), np.array(y_list)

X_val_seq, y_val_seq = build_sequences_for_val(
    df_train, df_val, feature_cols, lstm_scaler, LOOKBACK
)

lstm_val_probs = lstm_model.predict(X_val_seq, verbose=0)

print(f'  LSTM val probs shape  : {lstm_val_probs.shape}')
print(f'  LSTM val targets      : {y_val_seq.shape}')

# ── Build meta-feature matrix for val set ─────────────────
# Align XGB and LGBM to match LSTM row count
n_lstm = len(lstm_val_probs)

meta_val = np.column_stack([
    xgb_val_probs[-n_lstm:,  0],   # XGB  P(Sell)
    xgb_val_probs[-n_lstm:,  1],   # XGB  P(Hold)
    lgbm_val_probs[-n_lstm:, 0],   # LGBM P(Sell)
    lgbm_val_probs[-n_lstm:, 1],   # LGBM P(Hold)
    lstm_val_probs[:, 0],          # LSTM P(Sell)
    lstm_val_probs[:, 1],          # LSTM P(Hold)
])

y_val_aligned = y_val_seq.astype(int)

# ── Final ensemble predictions ────────────────────────────
val_preds = meta_model.predict(meta_val)
val_probs = meta_model.predict_proba(meta_val)

# ── Evaluation ────────────────────────────────────────────
f1_mac  = f1_score(y_val_aligned, val_preds, average='macro')
f1_wtd  = f1_score(y_val_aligned, val_preds, average='weighted')
bal_acc = balanced_accuracy_score(y_val_aligned, val_preds)

print(f'\nEnsemble Validation Performance (2024)')
print('=' * 50)
print(f'  F1 Macro          : {f1_mac:.4f}')
print(f'  F1 Weighted       : {f1_wtd:.4f}')
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'\nPer-class report:')
print(classification_report(
    y_val_aligned, val_preds,
    target_names=['Sell', 'Hold', 'Buy']
))

# ── Sanity checks ─────────────────────────────────────────
assert not np.any(np.isnan(meta_val)), \
    'NaN in val meta-features'
assert len(val_preds) == len(y_val_aligned), \
    'Prediction length mismatch'

print('Validation evaluation complete')

Base models loaded
  XGBoost  : ../models/xgb_model.pkl
  LightGBM : ../models/lgbm_model.pkl
  LSTM     : ../models/lstm_model.keras
  Lookback : 20 days

Tree model val predictions
  XGB  val probs shape  : (4920, 3)
  LGBM val probs shape  : (4920, 3)
  LSTM val probs shape  : (4920, 3)
  LSTM val targets      : (4920,)

Ensemble Validation Performance (2024)
  F1 Macro          : 0.3514
  F1 Weighted       : 0.4049
  Balanced Accuracy : 0.3763

Per-class report:
              precision    recall  f1-score   support

        Sell       0.25      0.16      0.19      1018
        Hold       0.50      0.80      0.62      2260
         Buy       0.41      0.17      0.24      1642

    accuracy                           0.46      4920
   macro avg       0.39      0.38      0.35      4920
weighted avg       0.42      0.46      0.40      4920

Validation evaluation complete


--------

### Cell 9 — Compare All 4 Models Side by Side

#### What We Do Here

Compare all 4 models on the same 4,920 validation
rows so the comparison is fully apples to apples.

  XGBoost   — flat features, gradient boosted trees

  LightGBM  — flat features, leaf-wise growth

  LSTM      — 20-day sequences, temporal patterns

  Ensemble  — meta-model stacking all three

#### Why Same Rows Matter

In Cell 8 the LSTM produced 4,920 sequences.

XGBoost and LightGBM also predicted on 4,920 rows.

All 4 models are evaluated on the exact same rows
so no model has an unfair advantage.

#### What We Are Looking For

  1. Does the ensemble beat the best base model?

  2. Which class does each model handle best?

  3. Where does each model fail?

#### What This Cell Does Not Do

It does not retune any model.

It does not go back and change anything.

It only reads and reports results.

The val set was opened in Cell 8 and stays open
only for reading — never for tuning.

In [9]:
# ============================================================
# CELL 9 - Compare All 4 Models Side by Side
# ============================================================
# Evaluate all 4 models on the same 4920 val rows
# Report F1 Macro, Balanced Accuracy, per-class breakdown
# ============================================================

# ── Generate base model predictions on same val rows ──────
# XGB and LGBM predict on flat features
# Use the same X_val rows that correspond to LSTM sequences
X_val_flat = df_val[feature_cols].values[-n_lstm:]
y_val_flat = df_val['target'].astype(int).values[-n_lstm:]

xgb_preds_val  = xgb_model.predict(X_val_flat)
lgbm_preds_val = lgbm_model.predict(X_val_flat)
lstm_preds_val = lstm_val_probs.argmax(axis=1)
# val_preds already has ensemble predictions from Cell 8

# ── Core metrics for all 4 models ─────────────────────────
models = [
    ('XGBoost',  xgb_preds_val),
    ('LightGBM', lgbm_preds_val),
    ('LSTM',     lstm_preds_val),
    ('Ensemble', val_preds),
]

print('All 4 Models — Validation Set (2024)')
print('=' * 60)
print(f'  {"Model":<12} {"F1 Macro":>10} '
      f'{"Bal Acc":>10} {"F1 Wtd":>10}')
print(f'  {"-" * 45}')

scores = {}
for name, preds in models:
    f1_mac  = f1_score(y_val_flat, preds, average='macro')
    f1_wtd  = f1_score(y_val_flat, preds, average='weighted')
    bal_acc = balanced_accuracy_score(y_val_flat, preds)
    scores[name] = {
        'f1_mac' : f1_mac,
        'f1_wtd' : f1_wtd,
        'bal_acc': bal_acc,
        'preds'  : preds
    }

best_f1 = max(v['f1_mac'] for v in scores.values())
for name, vals in scores.items():
    winner = '✅' if vals['f1_mac'] == best_f1 else '  '
    print(f'  {name:<12} {vals["f1_mac"]:>10.4f} '
          f'{vals["bal_acc"]:>10.4f} '
          f'{vals["f1_wtd"]:>10.4f} {winner}')

# ── Per class breakdown ────────────────────────────────────
print(f'\nPer-Class F1 Score Breakdown')
print('=' * 60)
print(f'  {"Model":<12} {"Sell F1":>10} '
      f'{"Hold F1":>10} {"Buy F1":>10}')
print(f'  {"-" * 45}')

for name, preds in models:
    from sklearn.metrics import f1_score as f1
    sell_f1 = f1(y_val_flat, preds, average=None)[0]
    hold_f1 = f1(y_val_flat, preds, average=None)[1]
    buy_f1  = f1(y_val_flat, preds, average=None)[2]
    print(f'  {name:<12} {sell_f1:>10.4f} '
          f'{hold_f1:>10.4f} {buy_f1:>10.4f}')

# ── Per class accuracy ────────────────────────────────────
print(f'\nPer-Class Recall (how many actual X were caught)')
print('=' * 60)
print(f'  {"Model":<12} {"Sell":>10} '
      f'{"Hold":>10} {"Buy":>10}')
print(f'  {"-" * 45}')

for name, preds in models:
    for cls, label in [(0,'Sell'),(1,'Hold'),(2,'Buy')]:
        pass
    recalls = []
    for cls in range(3):
        mask    = y_val_flat == cls
        correct = (preds[mask] == cls).sum()
        recalls.append(correct / mask.sum() * 100)
    print(f'  {name:<12} {recalls[0]:>9.1f}% '
          f'{recalls[1]:>9.1f}% {recalls[2]:>9.1f}%')

# ── Final summary ─────────────────────────────────────────
print(f'\nSummary')
print('=' * 60)
best_model = max(scores.items(), key=lambda x: x[1]['f1_mac'])
print(f'  Best model        : {best_model[0]}')
print(f'  Best F1 Macro     : {best_model[1]["f1_mac"]:.4f}')
print(f'  Ensemble F1 Macro : {scores["Ensemble"]["f1_mac"]:.4f}')
gap = scores['Ensemble']['f1_mac'] - best_model[1]['f1_mac']
if gap >= 0:
    print(f'  Ensemble gain     : +{gap:.4f} over best base model')
else:
    print(f'  Ensemble gap      : {gap:.4f} vs best base model')
    print(f'  Note: base models have stronger individual signals')
    print(f'        ensemble reflects honest stacking result')

print('\nComparison complete')

All 4 Models — Validation Set (2024)
  Model          F1 Macro    Bal Acc     F1 Wtd
  ---------------------------------------------
  XGBoost          0.3765     0.3799     0.4176 ✅
  LightGBM         0.3711     0.3742     0.4146   
  LSTM             0.3462     0.3785     0.3917   
  Ensemble         0.3514     0.3763     0.4049   

Per-Class F1 Score Breakdown
  Model           Sell F1    Hold F1     Buy F1
  ---------------------------------------------
  XGBoost          0.2346     0.5602     0.3346
  LightGBM         0.2163     0.5609     0.3362
  LSTM             0.2500     0.6114     0.1771
  Ensemble         0.1932     0.6179     0.2430

Per-Class Recall (how many actual X were caught)
  Model              Sell       Hold        Buy
  ---------------------------------------------
  XGBoost           24.4%      61.2%      28.4%
  LightGBM          21.8%      61.4%      29.0%
  LSTM              24.3%      78.0%      11.3%
  Ensemble          15.8%      79.8%      17.3%

Summary

-------

### Cell 10 — Save Meta-Model and Predictions

#### What We Save Here

Four files to ../models/ directory:

  meta_model.pkl        — trained Logistic Regression stacker
                          loaded by notebook 08 dashboard
                          for live ensemble predictions

  meta_val_probs.npy    — ensemble probabilities on 2024 val set
                          shape (4920, 3)
                          used for dashboard calibration reference

  meta_val_preds.npy    — final predicted labels on 2024 val set
                          shape (4920,)
                          used for performance reporting

  meta_val_targets.npy  — true labels for the 4920 val rows
                          shape (4920,)
                          used alongside preds for any
                          downstream analysis

#### Why We Save Targets Alongside Predictions

The dashboard needs to show historical accuracy.
Saving targets and predictions together means
notebook 08 does not need to reload and reprocess
the entire feature CSV just to show a score.

#### Verify Every File After Saving

A corrupted save discovered now saves hours of
debugging in notebook 08. Always reload and check.

In [10]:
# ============================================================
# CELL 10 - Save Meta-Model and Predictions
# ============================================================
# Save meta-model and val set predictions to disk
# Verify every file immediately after saving
# ============================================================

print('Saving Notebook 07 outputs')
print('=' * 55)

# ── File paths ────────────────────────────────────────────
PATH_META_MODEL   = f'{MODELS_DIR}meta_model.pkl'
PATH_VAL_PROBS    = f'{MODELS_DIR}meta_val_probs.npy'
PATH_VAL_PREDS    = f'{MODELS_DIR}meta_val_preds.npy'
PATH_VAL_TARGETS  = f'{MODELS_DIR}meta_val_targets.npy'

# ── Save meta-model ───────────────────────────────────────
joblib.dump(meta_model, PATH_META_MODEL)
size_model = os.path.getsize(PATH_META_MODEL) / 1024
print(f'  meta_model.pkl      saved : {size_model:>8.1f} KB')

# ── Save val probabilities ────────────────────────────────
np.save(PATH_VAL_PROBS, val_probs)
size_probs = os.path.getsize(PATH_VAL_PROBS) / 1024
print(f'  meta_val_probs.npy  saved : {size_probs:>8.1f} KB')

# ── Save val predictions ──────────────────────────────────
np.save(PATH_VAL_PREDS, val_preds)
size_preds = os.path.getsize(PATH_VAL_PREDS) / 1024
print(f'  meta_val_preds.npy  saved : {size_preds:>8.1f} KB')

# ── Save val targets ──────────────────────────────────────
np.save(PATH_VAL_TARGETS, y_val_aligned)
size_tgts = os.path.getsize(PATH_VAL_TARGETS) / 1024
print(f'  meta_val_targets.npy saved: {size_tgts:>8.1f} KB')

print()

# ── Verify all saved files ────────────────────────────────
print('Verifying saved files...')
print('-' * 55)

# Verify meta-model
meta_loaded      = joblib.load(PATH_META_MODEL)
orig_preds       = meta_model.predict(meta_val[:10])
loaded_preds     = meta_loaded.predict(meta_val[:10])
assert np.array_equal(orig_preds, loaded_preds), \
    'META MODEL VERIFY FAILED: predictions differ'
print(f'  meta_model.pkl           : verified ✅'
      f'  (predictions match on 10 samples)')

# Verify val probabilities
probs_loaded = np.load(PATH_VAL_PROBS)
assert probs_loaded.shape == val_probs.shape, \
    'VAL PROBS VERIFY FAILED: shape mismatch'
assert np.allclose(probs_loaded, val_probs), \
    'VAL PROBS VERIFY FAILED: values differ'
print(f'  meta_val_probs.npy       : verified ✅'
      f'  shape {probs_loaded.shape}')

# Verify val predictions
preds_loaded = np.load(PATH_VAL_PREDS)
assert preds_loaded.shape == val_preds.shape, \
    'VAL PREDS VERIFY FAILED: shape mismatch'
assert np.array_equal(preds_loaded, val_preds), \
    'VAL PREDS VERIFY FAILED: values differ'
print(f'  meta_val_preds.npy       : verified ✅'
      f'  shape {preds_loaded.shape}')

# Verify val targets
tgts_loaded = np.load(PATH_VAL_TARGETS)
assert tgts_loaded.shape == y_val_aligned.shape, \
    'VAL TARGETS VERIFY FAILED: shape mismatch'
assert np.array_equal(tgts_loaded, y_val_aligned), \
    'VAL TARGETS VERIFY FAILED: values differ'
print(f'  meta_val_targets.npy     : verified ✅'
      f'  shape {tgts_loaded.shape}')

print()

# ── Final summary ─────────────────────────────────────────
print('=' * 55)
print('NOTEBOOK 07 COMPLETE')
print('=' * 55)
print()
print('Files saved to ../models/:')
print(f'  meta_model.pkl           ← load in notebook 08')
print(f'  meta_val_probs.npy       ← dashboard reference')
print(f'  meta_val_preds.npy       ← dashboard reporting')
print(f'  meta_val_targets.npy     ← dashboard reporting')
print()
print('Complete Model Scorecard:')
print(f'  {"Model":<12} {"F1 Macro":>10} '
      f'{"Bal Acc":>10} {"F1 Wtd":>10}')
print(f'  {"-" * 45}')
for name, vals in scores.items():
    winner = ' ✅' if name == 'XGBoost' else '   '
    print(f'  {name:<12} {vals["f1_mac"]:>10.4f} '
          f'{vals["bal_acc"]:>10.4f} '
          f'{vals["f1_wtd"]:>10.4f}{winner}')
print()
print('Next : notebook 08 — Streamlit Dashboard')
print('  Load : meta_model.pkl')
print('  Load : xgb_model.pkl')
print('  Load : lgbm_model.pkl')
print('  Load : lstm_model.keras')
print('  Load : lstm_scaler.pkl')
print('  Load : lstm_sequence_config.pkl')
print('  Load : feature_cols.pkl')
print('=' * 55)

Saving Notebook 07 outputs
  meta_model.pkl      saved :      1.0 KB
  meta_val_probs.npy  saved :    115.4 KB
  meta_val_preds.npy  saved :     38.6 KB
  meta_val_targets.npy saved:     38.6 KB

Verifying saved files...
-------------------------------------------------------
  meta_model.pkl           : verified ✅  (predictions match on 10 samples)
  meta_val_probs.npy       : verified ✅  shape (4920, 3)
  meta_val_preds.npy       : verified ✅  shape (4920,)
  meta_val_targets.npy     : verified ✅  shape (4920,)

NOTEBOOK 07 COMPLETE

Files saved to ../models/:
  meta_model.pkl           ← load in notebook 08
  meta_val_probs.npy       ← dashboard reference
  meta_val_preds.npy       ← dashboard reporting
  meta_val_targets.npy     ← dashboard reporting

Complete Model Scorecard:
  Model          F1 Macro    Bal Acc     F1 Wtd
  ---------------------------------------------
  XGBoost          0.3765     0.3799     0.4176 ✅
  LightGBM         0.3711     0.3742     0.4146   
  LSTM     